# Cheap-Talk Benchmark — Kaggle Runner

Runs the full Sabani/Georgousis-aligned sweep (5 runs × 16 rounds × {PD,SH} × {no_comm,cheap_talk}) for ONE model on Kaggle's free 30 h/week T4 GPU.

**Before running:**
1. Settings (right panel) → **Accelerator** → `GPU T4 x2` (or `GPU P100`)
2. Settings → **Internet** → `On` (needed to download model weights)
3. Add Kaggle Secret named `HF_TOKEN` if you'll use a gated model (Llama, Gemma)
4. Edit the `MODEL` and `GITHUB_REPO` variables in cell 3 below

## Cell 1 — Install GPU inference deps

In [ ]:
!pip install -q transformers accelerate bitsandbytes openai python-dotenv

import torch
assert torch.cuda.is_available(), "GPU not enabled! Settings → Accelerator → GPU T4 x2"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2 — Clone the benchmark code from GitHub

In [ ]:
GITHUB_REPO = GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"

import os
if not os.path.exists('/kaggle/working/cheaptalk_bench'):
    !git clone $GITHUB_REPO /kaggle/working/cheaptalk_bench
%cd /kaggle/working/cheaptalk_bench
!ls

## Cell 3 — Load HF token from Kaggle Secrets (only needed for gated models)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded from Kaggle Secrets')
except Exception as e:
    print(f'No HF_TOKEN secret found: {e}')
    print('That is fine for non-gated models (Qwen). For Llama/Gemma:')
    print('  Kaggle → Settings → Secrets → Add new secret named HF_TOKEN')

## Cell 4 — Pick a model and run the full sweep

Tip: smoke-test with `--quick` (2 runs × 8 rounds, ~5 min) before committing to the full 5×16 sweep (~1–2 h on T4 for an 8B model).

In [ ]:
# --- pick ONE of these per Kaggle session (model gets loaded once) ---

# NOT gated -- start here
MODEL = "Qwen/Qwen2.5-7B-Instruct"
# MODEL = "Qwen/Qwen2.5-3B-Instruct"
# MODEL = "Qwen/Qwen2.5-14B-Instruct"   # ~9GB VRAM in 4-bit, fits T4
# MODEL = "Qwen/Qwen3-4B"
# MODEL = "Qwen/Qwen3-8B"
# MODEL = "Qwen/Qwen3-14B"

# Gated by Google -- accept license at the model's HF page first
# MODEL = "google/gemma-2-2b-it"
# MODEL = "google/gemma-2-9b-it"
# MODEL = "google/gemma-3-4b-it"

# Gated by Meta -- request access at the model's HF page first (~24h approval)
# MODEL = "meta-llama/Llama-3.1-8B-Instruct"

OUT_DIR = f"results/{MODEL.split('/')[-1]}"
print(f"Will run sweep for {MODEL} -> {OUT_DIR}")

In [ ]:
# Smoke test first (5-10 min). Comment out once you trust the setup.
!python run_full_sweep.py --provider local --model-id $MODEL --out-dir $OUT_DIR --quick --no-probe

In [ ]:
# Full Sabani-aligned sweep: 5 runs x 16 rounds x {pd,sh} x {no_comm,cheap_talk}
# Total runs: 20 (= 4 conditions x 5). Expected wall-time on T4:
#   3-4B model:  ~30 min
#   7-9B model:  ~1.5-2 h
#   14B model:   ~3 h
!python run_full_sweep.py --provider local --model-id $MODEL --out-dir $OUT_DIR --no-probe

## Cell 5 — Pack results for download

In [ ]:
import shutil
model_short = MODEL.split('/')[-1]
zip_path = f"/kaggle/working/results_{model_short}"
shutil.make_archive(zip_path, 'zip', OUT_DIR)
print(f"Done. Zip created at {zip_path}.zip")
print("Download from the right panel: Output -> results_*.zip")
!ls -lh /kaggle/working/*.zip

## Cell 6 — Quick on-Kaggle analysis (optional)

In [ ]:
!python analysis.py --results-dir $OUT_DIR